<a href="https://colab.research.google.com/github/muammarzainza2000/DataScience_240401010304_Muammar-Zain-Z.A/blob/main/Pertemuan10_Muammar_Zain_Z_A_240401010304.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
print(df["Churn"].value_counts(normalize=True))
print(df.dtypes)

(7043, 21)
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


In [2]:
from sklearn.model_selection import train_test_split

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

df = df.drop(columns=["customerID"])

X = pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True)
y = df["Churn"].map({"Yes": 1, "No": 0})

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [3]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [4]:
from sklearn.metrics import classification_report, roc_auc_score

pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1]

print(classification_report(y_te, pred))
print("ROC-AUC:", roc_auc_score(y_te, proba))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC: 0.8246208891988943


In [5]:
import numpy as np

hasil = pd.DataFrame({
    "Probabilitas_Churn": proba,
    "Prediksi": pred,
    "Aktual": y_te.values
})
print(hasil.head(10))

   Probabilitas_Churn  Prediksi  Aktual
0            0.000000         0       0
1            0.786667         1       0
2            0.090000         0       0
3            0.280000         0       0
4            0.000000         0       0
5            0.416667         0       0
6            0.393333         0       0
7            0.110000         0       0
8            0.006667         0       0
9            0.460000         0       1


In [6]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False).head(10))

TotalCharges                      0.177844
tenure                            0.164403
MonthlyCharges                    0.151054
Contract_Two year                 0.059944
InternetService_Fiber optic       0.042323
PaymentMethod_Electronic check    0.036455
Contract_One year                 0.029412
OnlineSecurity_Yes                0.028447
gender_Male                       0.025604
PaperlessBilling_Yes              0.024087
dtype: float64


## **Kesimpulan**
Model Random Forest memperoleh akurasi 79%. Namun pada kelas churn hanya memiliki recall 0,50 dan precision 0,63. Ini menujukkan bahwa model masoih melewatkan pelanggan yang sebenarnya churn. ROC-AUC sebesar 0,82 mengindikasikan kemampuan pemeringkatan model cukup baik, sehingga recall berpotensi ditingkatkan lewat threshold tuning. Feature Importance, TotalCharges, tenure dan MonthlyCharges menjadi fitur yang paling berpengaruh terhadap prediksi churn.